# S01 · Fundamentos, adquisición y procesamiento por fragmentos
**Aula alternativa · 150 minutos propuestos.** No sustituye el curso oficial.
Abre [la presentación de revisión](https://raw.githack.com/jazaineam1/BigData2026/mejora-pedagogica-integral/revision/s01.html). No necesitas internet para la ruta principal una vez abierto este cuaderno: incluye una copia del corpus de 200 procesos del repositorio, con su procedencia. No requiere secretos, instalar motores ni descargar archivos adicionales. Los datos son registros SECOP del snapshot, no una muestra representativa del universo.

**Objetivos observables.** Distinguir unidad de observación y formato; diagnosticar restricciones de volumen y velocidad; comprobar calidad e identidad; adquirir por páginas de forma reproducible; comparar agregaciones completas y por fragmentos; producir y releer un paquete con RAW, derivados y manifiesto. La consulta viva es opcional y se etiqueta separadamente.

**Cómo trabajar:** predice antes de ejecutar, observa las salidas, cambia un parámetro por vez y explica el límite de la evidencia. Las celdas A0–A8 están en orden. Ejecutar todas con configuración inicial produce un paquete local sin acceso externo; las variantes del reto son formativas. El resultado no certifica aprendizaje hasta comprobar tu propia modificación.


In [ ]:
import sys, json, csv, hashlib, math, tempfile, time, re, unicodedata
from collections import Counter
from pathlib import Path
import pandas as pd
from IPython.display import display
print("Python",sys.version.split()[0],"pandas",pd.__version__)
def sha(b): return hashlib.sha256(b).hexdigest()
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"),allow_nan=False).encode("utf-8")


## A1 · Diagnóstico, escala y presupuesto
Volumen mide cantidad; velocidad, ritmo de llegada; variedad, formas de representación; veracidad, calidad; valor, la capacidad de responder una pregunta. Estas dimensiones se relacionan: una fuente grande pero estática puede procesarse por lotes y una fuente pequeña de alta frecuencia puede acumular cola.

**Predice:** si llegan 120 registros/minuto y podemos procesar 80 durante cinco minutos, ¿cuántos quedan pendientes? Si cada uno ocupa 500 bytes, ¿cuánto ocupa la cola bruta? La cola no puede ser negativa. Para memoria, separa bytes de archivo comprimido y bytes mínimos de una matriz numérica. El cálculo no incluye índices, intérprete ni copias.

**Cómo se lee:** compara llegadas, completados y pendientes; cambia una tasa. **Qué nos dice:** la capacidad de servicio limita la salida. **Qué no permite concluir:** rendimiento de una máquina real. **Error común:** confundir registros por segundo con duración de una petición.


In [ ]:
llegada, capacidad, minutos = 120,80,5
recibidos=llegada*minutos
procesados=min(llegada,capacidad)*minutos
pendientes=recibidos-procesados
assert (recibidos,procesados,pendientes)==(600,400,200)
print({"recibidos":recibidos,"procesados":procesados,"pendientes":pendientes,"bytes_cola":pendientes*500})
for n,d in [(100_000,8),(1_000_000,128),(10_000_000,128)]:
    print(n,d,"mínimo GiB",n*d*8/2**30)


## A2 · Grano y representación
Una fila de procesos no equivale a una fila de contratos. Cada contrato referencia un proceso; un proceso puede tener varios contratos. El siguiente ejemplo es ficticio y no se mezcla con SECOP. Un left join conserva procesos sin coincidencia, mientras un inner join los descarta. Si la dimensión repite identidades, la unión puede multiplicar filas.

**Predice:** P1 tiene dos contratos, P2 uno y P3 ninguno. ¿Cuántos procesos distintos hay tras la unión izquierda? ¿Cuántas filas resultan? La salida contiene cuatro filas y tres procesos, no cuatro procesos. Comprueba la identidad contractual no nula antes de sumar valores.

En CSV, un registro aparece en una fila y las columnas tienen delimitadores; JSON permite anidar listas. Serializar no es validar. Un archivo puede leerse correctamente pero representar el grano equivocado. En el producto RAW de esta sesión conservaremos procesos, no asignaremos dinero a ellos.

La siguiente extensión de formatos serializa dos observaciones ficticias con comas, comillas y caracteres acentuados. Compara bytes y tipos recuperados. CSV no impone tipos: un número leído como texto requiere un contrato. JSON preserva tipos básicos, XML utiliza texto y etiquetas; ninguno garantiza significado.


In [ ]:
procesos=pd.DataFrame({"id_proceso":["P1","P2","P3"]})
contratos=pd.DataFrame({"id_contrato":["C1","C2","C3"],"id_proceso":["P1","P1","P2"]})
izq=procesos.merge(contratos,on="id_proceso",how="left",validate="one_to_many")
int_=procesos.merge(contratos,on="id_proceso",how="inner",validate="one_to_many")
assert len(izq)==4 and izq.id_proceso.nunique()==3 and izq.id_contrato.count()==3
assert len(int_)==3 and int_.id_proceso.nunique()==2
display(izq)
print("Inner:",len(int_),"filas,",int_.id_proceso.nunique(),"procesos")


In [ ]:
import io, xml.etree.ElementTree as ET
ejemplo=[{"id":"P1","entidad":"Bogotá, D.C.","cantidad":2},
         {"id":"P2","entidad":'Entidad "B"',"cantidad":0}]
buf=io.StringIO()
escritor=csv.DictWriter(buf,fieldnames=list(ejemplo[0]))
escritor.writeheader(); escritor.writerows(ejemplo)
texto_csv=buf.getvalue()
leido_csv=list(csv.DictReader(io.StringIO(texto_csv)))
assert leido_csv[0]["cantidad"]=="2"
texto_json=json.dumps(ejemplo,ensure_ascii=False)
assert json.loads(texto_json)==ejemplo
raiz=ET.Element("procesos")
for registro in ejemplo:
    nodo=ET.SubElement(raiz,"proceso")
    for k,v in registro.items(): ET.SubElement(nodo,k).text=str(v)
texto_xml=ET.tostring(raiz,encoding="utf-8")
assert [x.findtext("id") for x in ET.fromstring(texto_xml)]==["P1","P2"]
for nombre,b in [("CSV",texto_csv.encode()),("JSON",texto_json.encode()),("XML",texto_xml)]:
    print(nombre,len(b),"bytes")
print("CSV recupera cantidad como texto; JSON como número.")


## A3 · Fuente real, calidad, pandas y texto
El snapshot se incorpora directamente para que el laboratorio funcione sin red. Fuente original indicada por el repositorio: Datos/bandeja_revision_2026.json; versión del archivo tc2_corpus_2026.json: blob 4b1ac0dd22137984b7e81864a70e04075af2af39. Los registros contienen ID de proceso, nombre, descripción, entidad, enlace, fuente y tipo. La copia preserva los objetos del snapshot; no es una descarga de SECOP en vivo. La celda de datos es extensa: se puede plegar; no es contenido explicativo.

Después de cargar, comprobamos esquema, unicidad y faltantes. La tabla agrupada cambia grano de proceso a entidad; size cuenta filas. Si se modifica una copia para introducir duplicados o faltantes, RAW no debe cambiar. La descripción se tokeniza por letras y números normalizados: un término puede aparecer varias veces en un proceso, por lo que ocurrencias y frecuencia documental son diferentes.

**Cómo se lee:** total de grupos, suma de conteos y primeras filas. **Qué nos dice:** cuántos procesos de esta fuente corresponden a cada entidad. **Qué no permite concluir:** concentración del gasto ni representatividad nacional. **Error común:** eliminar registros por tener textos iguales aunque sus IDs sean diferentes.


In [ ]:
# Copia de los 200 registros SECOP del snapshot del repositorio. No editar.
CORPUS = json.loads("[{\"id_proceso\":\"CO1.REQ.10017413\",\"nombre_proceso\":\"CONTRATO INTERADMINITRATIVO PARA EL FORTALECIMIENTO DEL NIVEL PRIMARIO; BASADO EN LA ATENCIÓN PRIMARIA EN SALUD A TRAVÉS DE LA CONFORMACIÓN \",\"descripcion\":\"CONTRATO INTERADMINITRATIVO PARA EL FORTALECIMIENTO DEL NIVEL PRIMARIO; BASADO EN LA ATENCIÓN PRIMARIA EN SALUD A TRAVÉS DE LA CONFORMACIÓN Y OPERACIÓN DE EQUIPOS BÁSICOS DE SALUD \",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9868453\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10021334\",\"nombre_proceso\":\"CONTRATO DE PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA FORTALECER LOS PROCESOS COMUNICACIONALES DE LA SECRETARÍA DE COMUNICACIONES; \",\"descripcion\":\"CONTRATO DE PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA FORTALECER LOS PROCESOS COMUNICACIONALES DE LA SECRETARÍA DE COMUNICACIONES; MEDIANTE EL DISEÑO; PRODUCCIÓN Y DIFUSIÓ\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9877717\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10027889\",\"nombre_proceso\":\"CI-0709-2026\",\"descripcion\":\"CI-0709-2026\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9879068\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10048613\",\"nombre_proceso\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA DE LA INSTITUCIÓN ETNOEDUCATIVA INTEGRAL RURAL KUISA; LA \",\"descripcion\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA DE LA INSTITUCIÓN ETNOEDUCATIVA INTEGRAL RURAL KUISA; LA INSTITUCION ETNOEDUCATIVA INTEGRAL RURAL\",\"entidad\":\"MUNICIPIO DE URIBIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9900290\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10056281\",\"nombre_proceso\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA DE LA INSTITUCION ETNOEDUCATIVA INTEGRAL RURAL JACOBO ARE\",\"descripcion\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA DE LA INSTITUCION ETNOEDUCATIVA INTEGRAL RURAL JACOBO ARENDS Y LA INSTITUCION ETNOEDUCATIVA INTEG\",\"entidad\":\"MUNICIPIO DE URIBIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9908043\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10083057\",\"nombre_proceso\":\"CONVENIO SOLIDARIO\",\"descripcion\":\"CONVENIO SOLIDARIO\",\"entidad\":\"GOBERNACION DE NARIÑO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9937124\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10246305\",\"nombre_proceso\":\"Prestar los servicios tecnológicos necesarios para la puesta en operación del software utilizado para la comisión de escrutinio nacional; as\",\"descripcion\":\"Prestar los servicios tecnológicos necesarios para la puesta en operación del software utilizado para la comisión de escrutinio nacional; asegurando los componentes requeridos; Inf\",\"entidad\":\"Consejo Nacional Electoral\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.10104026\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1109601\",\"nombre_proceso\":\"SECRETARIA DISTRITAL DE MOVILIDAD\",\"descripcion\":\"SECRETARIA DISTRITAL DE MOVILIDAD\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1144334\",\"nombre_proceso\":\"CONTRATO TEVEANDINA- PRENSA\",\"descripcion\":\"CONTRATO TEVEANDINA- PRENSA\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1106024\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1227580\",\"nombre_proceso\":\"CONTRATO IVA\",\"descripcion\":\"CONTRATO IVA\",\"entidad\":\"PROSPERIDAD SOCIAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1188491\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1229269\",\"nombre_proceso\":\"IMPRENTA NACIONAL DE COLOMBIA\",\"descripcion\":\"IMPRENTA NACIONAL DE COLOMBIA\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1189868\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1235779\",\"nombre_proceso\":\"CONTRATO DE ARRENDAMIENTO\",\"descripcion\":\"CONTRATO DE ARRENDAMIENTO\",\"entidad\":\"(Secretaría Distrital de Integración Social)\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1206076\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1253135\",\"nombre_proceso\":\"PONER A DISPOSICIÓN DE SUSCRIPTORES LÍNEAS MÓVILES PREPAGO SERVICIOS TIC - EMERGENCIA SANITARIA; ECONÓMICA Y ECOLÓGICA COVID-19 - COMUNICACI\",\"descripcion\":\"PONER A DISPOSICIÓN DE SUSCRIPTORES LÍNEAS MÓVILES PREPAGO SERVICIOS TIC - EMERGENCIA SANITARIA; ECONÓMICA Y ECOLÓGICA COVID-19 - COMUNICACIÓN CELULAR S.A. COMCEL S.A.\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1213079\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1257693\",\"nombre_proceso\":\"SUMINISTRO E INSTALACIÓN DE EQUIPAMENTO BIOMEDICO PARA EL HOSPITAL DEPARTAMENTAL DE VILLAVICENCIO E.S.E. COVID 19\",\"descripcion\":\"SUMINISTRO E INSTALACIÓN DE EQUIPAMENTO BIOMEDICO PARA EL HOSPITAL DEPARTAMENTAL DE VILLAVICENCIO E.S.E. COVID 19\",\"entidad\":\"DEPARTAMENTO DEL META\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1218081\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1309251\",\"nombre_proceso\":\"CONTRATACIÓN DIRECTA - URGENCIA MANIFIESTA\",\"descripcion\":\"CONTRATACIÓN DIRECTA - URGENCIA MANIFIESTA\",\"entidad\":\"INSTITUTO DE DESARROLLO URBANO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1268026\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1423544\",\"nombre_proceso\":\"FORTALECIMIENTO A LA PRODUCTIVIDAD DEL SECTOR AGROPECUARIO EN SITUACIÓN DE  EMERGENCIA SANITARIA DEL DEPARTAMENTO DE CORDOBA; IDENTIFICADO C\",\"descripcion\":\"FORTALECIMIENTO A LA PRODUCTIVIDAD DEL SECTOR AGROPECUARIO EN SITUACIÓN DE  EMERGENCIA SANITARIA DEL DEPARTAMENTO DE CORDOBA; IDENTIFICADO CON EL CÓDIGO BPIN  2020000020008\",\"entidad\":\"DEPARTAMENTO DE CORDOBA\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1561617\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO CONECTIVIDAD\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO CONECTIVIDAD\",\"entidad\":\"GOBERNACION DE SUCRE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1513764\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1591012\",\"nombre_proceso\":\"PROVEER SOLUCIONES DE VIVIENDA EN EL MUNICIPIO DE MARIA LA BAJA; DEPARTAMENTO DE BOLÍVAR - CONSORCIO BOLÍVAR\",\"descripcion\":\"PROVEER SOLUCIONES DE VIVIENDA EN EL MUNICIPIO DE MARIA LA BAJA; DEPARTAMENTO DE BOLÍVAR - CONSORCIO BOLÍVAR\",\"entidad\":\"FONDO ADAPTACIÓN\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1540718\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1676905\",\"nombre_proceso\":\"Aunar esfuerzos técnicos; administrativos y financieros entre el Ministerio del Deporte y el Municipio de Natagaima; para la ejecución del p\",\"descripcion\":\"Aunar esfuerzos técnicos; administrativos y financieros entre el Ministerio del Deporte y el Municipio de Natagaima; para la ejecución del proyecto denominado MEJORAMIENTO Y ADECU\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1625113\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1681646\",\"nombre_proceso\":\"CONTRATO DE ORDEN DE SERVICIO\",\"descripcion\":\"CONTRATO DE ORDEN DE SERVICIO\",\"entidad\":\"GOBERNACIÓN DE LA GUAJIRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1629766\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1766696\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1713178\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1791463\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS - BANCO DE LA REPUBLICA\",\"descripcion\":\"PRESTACION DE SERVICIOS - BANCO DE LA REPUBLICA\",\"entidad\":\"MINISTERIO DE HACIENDA Y CREDITO PUBLICO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1738021\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1828218\",\"nombre_proceso\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA\",\"descripcion\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA\",\"entidad\":\"MINISTERIO DE EDUCACION NACIONAL (MEN)\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1779662\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1842719\",\"nombre_proceso\":\"LA SECRETARÍA DISTRITAL DE MOVILIDAD Y LA POLICÍA NACIONAL; AÚNAN ESFUERZOS PARA COORDINAR Y COOPERAR MUTUAMENTE PARA EJERCER EL CONTROL Y R\",\"descripcion\":\"LA SECRETARÍA DISTRITAL DE MOVILIDAD Y LA POLICÍA NACIONAL; AÚNAN ESFUERZOS PARA COORDINAR Y COOPERAR MUTUAMENTE PARA EJERCER EL CONTROL Y REGULACIÓN DEL TRÁNSITO Y EL TRANSPORTE E\",\"entidad\":\"SECRETARÍA DISTRITAL DE MOVILIDAD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1801312\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1859670\",\"nombre_proceso\":\"FAMOC DEPANEL S.A.\",\"descripcion\":\"FAMOC DEPANEL S.A.\",\"entidad\":\"SECRETARIA DISTRITAL DE SEGURIDAD, CONVIVENCIA Y JUSTICIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1808987\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1862814\",\"nombre_proceso\":\"OICH\",\"descripcion\":\"OICH\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1807608\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1903338\",\"nombre_proceso\":\"Convenio interadministrativo\",\"descripcion\":\"Convenio interadministrativo\",\"entidad\":\"GOBERNACION DE NARIÑO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1847239\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1928260\",\"nombre_proceso\":\"Caja Colombiana de Subsidio Familiar  COLSUBSIDIO\",\"descripcion\":\"Caja Colombiana de Subsidio Familiar  COLSUBSIDIO\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1941822\",\"nombre_proceso\":\"AUNAR ESFUERZOS PARA ASEGURAR LA PRESTACIÓN DEL SERVICIO PÚBLICO ESENCIAL DEATENCIÓN Y CONTROL DE INCENDIOS Y DEMÁS ACTIVIDADES CONEXAS\",\"descripcion\":\"AUNAR ESFUERZOS PARA ASEGURAR LA PRESTACIÓN DEL SERVICIO PÚBLICO ESENCIAL DEATENCIÓN Y CONTROL DE INCENDIOS Y DEMÁS ACTIVIDADES CONEXAS\",\"entidad\":\"ALCALDÍA DE POPAYÁN\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1885416\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1942046\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO DE ADMINISTRACIÓN DELEGADA ENTRE EL MUNICIPIO DE ITAGÜÍ Y LA AGENCIA DE DESARROLLO LOCAL DE ITAGÜÍ ADELI PARA \",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO DE ADMINISTRACIÓN DELEGADA ENTRE EL MUNICIPIO DE ITAGÜÍ Y LA AGENCIA DE DESARROLLO LOCAL DE ITAGÜÍ ADELI PARA LA ATENCIÓN PREVENTIVA Y CORRECTIVA Y AD\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1885828\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1949451\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA POLÍTICA DE EST\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA POLÍTICA DE ESTADO PARA EL DESARROLLO INTEGRAL DE LA PR\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1893361\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1949570\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA POLÍTICA DE EST\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA POLÍTICA DE ESTADO PARA EL DESARROLLO INTEGRAL DE LA PR\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1893802\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1958141\",\"nombre_proceso\":\"APOYO A LA GESTIÓN PRIMERA INFANCIA\",\"descripcion\":\"APOYO A LA GESTIÓN PRIMERA INFANCIA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1901948\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1958230\",\"nombre_proceso\":\"REALIZAR LA GERENCIA INTEGRAL Y COORDINACION PARA REALIZAR EL MEJORAMIENTO DE LOS SISTEMAS DE SONIDO E ILUMINACIÓN DEL ESTADIO METROPOLITANO\",\"descripcion\":\"REALIZAR LA GERENCIA INTEGRAL Y COORDINACION PARA REALIZAR EL MEJORAMIENTO DE LOS SISTEMAS DE SONIDO E ILUMINACIÓN DEL ESTADIO METROPOLITANO ROBERTO MELÉNDEZ BARRANQUILLA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1902364\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2052103\",\"nombre_proceso\":\"CONVENIO CONV-SED-016-2021\",\"descripcion\":\"CONVENIO CONV-SED-016-2021\",\"entidad\":\"GOBERNACION DE SUCRE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1998706\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2094652\",\"nombre_proceso\":\"Universidad Distrital Francisco José de Caldas\",\"descripcion\":\"Universidad Distrital Francisco José de Caldas\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2136149\",\"nombre_proceso\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE VALENCIA; CÓRDOBA\",\"descripcion\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE VALENCIA; CÓRDOBA\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2078925\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2197504\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO PARA LA INTERVENTORÍA DE OBRAS DE INFRAESTRUCTURA DEL MUNICIPIO DE RIONEGRO.\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO PARA LA INTERVENTORÍA DE OBRAS DE INFRAESTRUCTURA DEL MUNICIPIO DE RIONEGRO.\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2138606\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2256136\",\"nombre_proceso\":\"CONVENIO ESPECÍFICO DE COOPERACIÓN\",\"descripcion\":\"CONVENIO ESPECÍFICO DE COOPERACIÓN\",\"entidad\":\"ALCALDIA DE PASTO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2210538\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2267818\",\"nombre_proceso\":\"CI-014-2021\",\"descripcion\":\"CI-014-2021\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2216573\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2273783\",\"nombre_proceso\":\"CONTRATACION DIRECTA FIDUCIARIA CORFICOLOMBIANA SA\",\"descripcion\":\"CONTRATACION DIRECTA FIDUCIARIA CORFICOLOMBIANA SA\",\"entidad\":\"TRANSMILENIO S.A.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2231919\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2282243\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2223381\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2306622\",\"nombre_proceso\":\"CONVENIO BECAS POLICIA NACIONAL\",\"descripcion\":\"CONVENIO BECAS POLICIA NACIONAL\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2247066\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2422511\",\"nombre_proceso\":\"ADQUISICIÓN; OPTIMIZACIÓN Y ACTUALIZACIÓN DEL LICENCIAMIENTO PARA LA PLATAFORMA DE RECEPCIÓN Y ANÁLISIS DE COMUNICACIONES INTERCEPTADAS CON \",\"descripcion\":\"ADQUISICIÓN; OPTIMIZACIÓN Y ACTUALIZACIÓN DEL LICENCIAMIENTO PARA LA PLATAFORMA DE RECEPCIÓN Y ANÁLISIS DE COMUNICACIONES INTERCEPTADAS CON EL PROTOCOLO DE COMUNICACIONES ETSI - II\",\"entidad\":\"DIRECCIÓN DE INVESTIGACIÓN CRIMINAL E INTERPOL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2362029\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2427611\",\"nombre_proceso\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE PUERTO LOPEZ PARA LA EJECUCIÓN DEL\",\"descripcion\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE PUERTO LOPEZ PARA LA EJECUCIÓN DEL PROYECTO DENOMINADO CONSTRUCCIÓN DE PLA\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2367035\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2435456\",\"nombre_proceso\":\"Aunar esfuerzos técnicos; administrativos y financieros para ejecutar el plan integral de ocupantes de espacio público (OEP) de la Primera L\",\"descripcion\":\"Aunar esfuerzos técnicos; administrativos y financieros para ejecutar el plan integral de ocupantes de espacio público (OEP) de la Primera Línea Del Metro De Bogotá Tramo 1; en su \",\"entidad\":\"METRO DE BOGOTA SA\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2435627\",\"nombre_proceso\":\"construcción de escenario polideportivo cancha sintetica en pie de pato cabecera municial del municipio de Alto Baudó Departamento del Choco\",\"descripcion\":\"construcción de escenario polideportivo cancha sintetica en pie de pato cabecera municial del municipio de Alto Baudó Departamento del Choco\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2378302\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2437231\",\"nombre_proceso\":\"SECRETARIA DISTRITAL DE INTEGRACION SOCIAL\",\"descripcion\":\"SECRETARIA DISTRITAL DE INTEGRACION SOCIAL\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2442640\",\"nombre_proceso\":\"CD-660-2021\",\"descripcion\":\"CD-660-2021\",\"entidad\":\"MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2380558\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2443494\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"GOBERNACIÓN DE BOYACÁ\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2384919\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2443796\",\"nombre_proceso\":\"AUNAR ESFUERZOS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE CLEMENCIA; BOLÍVAR; PARA LA CONSTRUCCIÓN DEL PROYECTO DENOMINADO  CONSTRUC\",\"descripcion\":\"AUNAR ESFUERZOS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE CLEMENCIA; BOLÍVAR; PARA LA CONSTRUCCIÓN DEL PROYECTO DENOMINADO  CONSTRUCCIÓN DE ESTADIO DE DE SOFTBOL EN EL MUNI\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2382614\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2454035\",\"nombre_proceso\":\"ADMINISTRACIÓN DELEGADA DE RECURSOS PARA LA OPTIMIZACIÓN Y CONSTRUCCIÓN DE LOS SISTEMAS DE ALCANTARILLADO EN LOS CENTROS POBLADOS TABLAZO; P\",\"descripcion\":\"ADMINISTRACIÓN DELEGADA DE RECURSOS PARA LA OPTIMIZACIÓN Y CONSTRUCCIÓN DE LOS SISTEMAS DE ALCANTARILLADO EN LOS CENTROS POBLADOS TABLAZO; PONTEZUELA; ABREO Y LA PLAYA DEL MUNICIPI\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2390852\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2454516\",\"nombre_proceso\":\"CI-031-2021\",\"descripcion\":\"CI-031-2021\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2395823\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2747342\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARIA DISTRITAL DE RECREACION Y DEPORTES PARA EL DESARROLLO DE ACTIVIDADES ENCAMINAD\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARIA DISTRITAL DE RECREACION Y DEPORTES PARA EL DESARROLLO DE ACTIVIDADES ENCAMINADAS A LA MASIFICACION Y PROMOCIÓN DEL DEP\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2673037\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2790843\",\"nombre_proceso\":\"CD-436-2022\",\"descripcion\":\"CD-436-2022\",\"entidad\":\"MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2716096\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.280655\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO AGENCIA NACIONAL INMOBILIARIA\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO AGENCIA NACIONAL INMOBILIARIA\",\"entidad\":\"SUPERINTENDENCIA DEL SUBSIDIO FAMILIAR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.272852\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2812254\",\"nombre_proceso\":\"CANAL REGIONAL DE TELEVISIÓN TEVEANDINA LTDA\",\"descripcion\":\"CANAL REGIONAL DE TELEVISIÓN TEVEANDINA LTDA\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2737344\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2899950\",\"nombre_proceso\":\"OICH\",\"descripcion\":\"OICH\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2829538\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3076649\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"ALCALDIA DE PASTO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2997588\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3100263\",\"nombre_proceso\":\"TECNOLOGIAS DE LA INFORMACION Y PAGO INTEGRADO S.A.S.\",\"descripcion\":\"TECNOLOGIAS DE LA INFORMACION Y PAGO INTEGRADO S.A.S.\",\"entidad\":\"SECRETARIA DISTRITAL DE SEGURIDAD, CONVIVENCIA Y JUSTICIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3020366\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3279289\",\"nombre_proceso\":\"CONTRATACION DIRECTA 836 DE 2022\",\"descripcion\":\"CONTRATACION DIRECTA 836 DE 2022\",\"entidad\":\"SUPERINTENDENCIA DE NOTARIADO Y REGISTRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3195558\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.339790\",\"nombre_proceso\":\"PROGRAMA DE LAS NACIONES UNIDAS PARA EL DESARROLLO\",\"descripcion\":\"PROGRAMA DE LAS NACIONES UNIDAS PARA EL DESARROLLO\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.330265\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3482395\",\"nombre_proceso\":\"CONTRATO DE EMPRÉSTITO\",\"descripcion\":\"CONTRATO DE EMPRÉSTITO\",\"entidad\":\"ALCALDIA DE PASTO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3403614\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3506421\",\"nombre_proceso\":\"CONVENIO DE COPERACIÓN INTERNACIONAL - PNUD - DAR\",\"descripcion\":\"CONVENIO DE COPERACIÓN INTERNACIONAL - PNUD - DAR\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3420839\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3533907\",\"nombre_proceso\":\"PLAZA MAYOR MEDELLÍN CONVENCIONES Y EXPOSICIONES S.A.\",\"descripcion\":\"PLAZA MAYOR MEDELLÍN CONVENCIONES Y EXPOSICIONES S.A.\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3450053\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3549233\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA ACOMPAÑAR A LA SECRETARÍA DE GOBIERNO DEL DISTRITO BARRANQUILLA EN LA ATENCIÓN DE COMUNID\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA ACOMPAÑAR A LA SECRETARÍA DE GOBIERNO DEL DISTRITO BARRANQUILLA EN LA ATENCIÓN DE COMUNIDADES NEGRAS AFROCOLOMBIANAS; RAIZALES Y \",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3463192\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3583869\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3498545\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3590455\",\"nombre_proceso\":\"Contrato de Emprestito Banco de Bogotá\",\"descripcion\":\"Contrato de Emprestito Banco de Bogotá\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3506491\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3615563\",\"nombre_proceso\":\"Prestar a la Federación Colombiana de Municipios en cumplimiento de la función pública asignada; por sus propios medios; con plena autonomía\",\"descripcion\":\"Prestar a la Federación Colombiana de Municipios en cumplimiento de la función pública asignada; por sus propios medios; con plena autonomía técnica; administrativa y operacional; \",\"entidad\":\"FEDERACION COLOMBIANA DE MUNICIPIOS\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3530681\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3684277\",\"nombre_proceso\":\"Entrega en comodatos espacio de lote ubicado en la calle 27  entre autopista sur y la carrera 41 Itagui para ser utilizado  Patios  de vehíc\",\"descripcion\":\"Entrega en comodatos espacio de lote ubicado en la calle 27  entre autopista sur y la carrera 41 Itagui para ser utilizado  Patios  de vehículos\",\"entidad\":\"AREA METROPOLITANA DEL VALLE DE ABURRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3687992\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3714823\",\"nombre_proceso\":\"EL SERVICIO DE MANTENIMIENTO INSPECCIÓN MAYOR ESTRUCTURAL PDM; CUMPLIMIENTO DE BOLETINES DE SERVICIO MANDATORIOS Y TRABAJOS ESPECIALES DE LA\",\"descripcion\":\"EL SERVICIO DE MANTENIMIENTO INSPECCIÓN MAYOR ESTRUCTURAL PDM; CUMPLIMIENTO DE BOLETINES DE SERVICIO MANDATORIOS Y TRABAJOS ESPECIALES DE LA AERONAVE C-130 FAC1018; DE ACUERDO CON \",\"entidad\":\"FUERZA AEROESPACIAL COLOMBIANA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3636264\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3730074\",\"nombre_proceso\":\"CONSTRUCCION DE LA FASE II DEL CENTRO DE INSTRUCCIÓN EN CAÍDA LIBRE E INVESTIGACIÓN AERONÁUTICA DE LA FAC EN LAS INSTALACIONES DEL MUSEO AER\",\"descripcion\":\"CONSTRUCCION DE LA FASE II DEL CENTRO DE INSTRUCCIÓN EN CAÍDA LIBRE E INVESTIGACIÓN AERONÁUTICA DE LA FAC EN LAS INSTALACIONES DEL MUSEO AEROESPACIAL EN TOCANCIPÁ - CUNDINAMARCA; C\",\"entidad\":\"FUERZA AEROESPACIAL COLOMBIANA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3644540\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3730428\",\"nombre_proceso\":\"NORMALIZACIÓN DE REDES ELÉCTRICAS BARRIO SOLIMAR\",\"descripcion\":\"NORMALIZACIÓN DE REDES ELÉCTRICAS BARRIO SOLIMAR\",\"entidad\":\"MINISTERIO DE MINAS Y ENERGIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3644468\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3755271\",\"nombre_proceso\":\"Contratación Directa\",\"descripcion\":\"Contratación Directa\",\"entidad\":\"POLITECNICO COLOMBIANO JAIME ISAZA CADAVID\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3668146\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3755377\",\"nombre_proceso\":\"SERVICIOS DE MANTENIMIENTO PROGRAMADO (RUTINA) Y MANTENIMIENTO NO PROGRAMADO (NO RUTINA) DE LAS AERONAVES KFIR DE LA FUERZA AÉREA COLOMBIANA\",\"descripcion\":\"SERVICIOS DE MANTENIMIENTO PROGRAMADO (RUTINA) Y MANTENIMIENTO NO PROGRAMADO (NO RUTINA) DE LAS AERONAVES KFIR DE LA FUERZA AÉREA COLOMBIANA; DE ACUERDO A ANEXO TÉCNICO\",\"entidad\":\"FUERZA AEROESPACIAL COLOMBIANA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3675471\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3763772\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA REFORZAR Y FORTALECER LAS ACTIVIDADES DE ASISTENCIA OPERATIVA Y APOYO LOGÍSTICO REQUERIDA\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA REFORZAR Y FORTALECER LAS ACTIVIDADES DE ASISTENCIA OPERATIVA Y APOYO LOGÍSTICO REQUERIDAS POR LA SECRETARÍA DE GOBIERNO MUNICIPA\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3683039\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3766622\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO - ARAUQUITA -ARAUCA\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO - ARAUQUITA -ARAUCA\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3678710\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3766925\",\"nombre_proceso\":\"CONVENIO DE COFINANCIACION - SACUDETE TIPO 1 - TOCA; BOYACA\",\"descripcion\":\"CONVENIO DE COFINANCIACION - SACUDETE TIPO 1 - TOCA; BOYACA\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3678790\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3920667\",\"nombre_proceso\":\"Aunar esfuerzos técnicos; administrativos y financieros para ejecutar el plan integral de ocupantes de espacio público (OEP) de la primera l\",\"descripcion\":\"Aunar esfuerzos técnicos; administrativos y financieros para ejecutar el plan integral de ocupantes de espacio público (OEP) de la primera línea del metro de Bogotá Tramo 1; en su \",\"entidad\":\"METRO DE BOGOTA SA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3827148\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3982234\",\"nombre_proceso\":\"FORTALECIMIENTO TÉCNICO DEL EJERCICIO DE AUTORIDAD AMBIENTAL  URBANO EN EL ÁREA METROPOLITANA DEL VALLE DE ABURRÁ\",\"descripcion\":\"FORTALECIMIENTO TÉCNICO DEL EJERCICIO DE AUTORIDAD AMBIENTAL  URBANO EN EL ÁREA METROPOLITANA DEL VALLE DE ABURRÁ\",\"entidad\":\"AREA METROPOLITANA DEL VALLE DE ABURRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3927703\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4038392\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO PARA EL APOYO A LA GESTIÓN DE SERVICIOS ESPECIALIZADOS PARA LAS ETAPAS DE EJECUCIÓN Y RECAUDO DE LA CONTRIBUCIÓ\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO PARA EL APOYO A LA GESTIÓN DE SERVICIOS ESPECIALIZADOS PARA LAS ETAPAS DE EJECUCIÓN Y RECAUDO DE LA CONTRIBUCIÓN DE VALORIZACIÓN PARA EL PROYECTO DE VA\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3944722\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4072320\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS PROFESIONALES\",\"descripcion\":\"PRESTACION DE SERVICIOS PROFESIONALES\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3977655\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4115029\",\"nombre_proceso\":\"ARRENDAMIENTO DE BIENES INMUEBLES PARA EL FUNCIONAMIENTO ADMINISTRATIVO; TECNICO Y OPERATIVO DE LAS DIFERENTES SECRETARIAS Y/O DEPENDENCIAS \",\"descripcion\":\"ARRENDAMIENTO DE BIENES INMUEBLES PARA EL FUNCIONAMIENTO ADMINISTRATIVO; TECNICO Y OPERATIVO DE LAS DIFERENTES SECRETARIAS Y/O DEPENDENCIAS EN EL DISTRITO DE BARRANQUILLA.\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4020262\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4134446\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO POR ADMINISTRACIÓN DELEGADA PARA APOYAR EL DESARROLLO DE ESTRATEGIAS ENCAMINADAS A PROMOVER EL FORTALECIMIENTO \",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO POR ADMINISTRACIÓN DELEGADA PARA APOYAR EL DESARROLLO DE ESTRATEGIAS ENCAMINADAS A PROMOVER EL FORTALECIMIENTO EMPRESARIAL; CRECIMIENTO DEL TURISMO; PR\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4039795\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4154601\",\"nombre_proceso\":\"CDPS CANASTA EDUCATIVA 012 DE 2023\",\"descripcion\":\"CDPS CANASTA EDUCATIVA 012 DE 2023\",\"entidad\":\"GOBERNACIÓN DE LA GUAJIRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4059614\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4157866\",\"nombre_proceso\":\"DESARROLLO DE LAS ACTIVIDADES DEL PLAN DE INTERVENCIONES COLECTIVAS (PIC); PARA LA PROMOCION DE LA SALUD Y PREVENCION DE LA ENFERMEDAD EN EL\",\"descripcion\":\"DESARROLLO DE LAS ACTIVIDADES DEL PLAN DE INTERVENCIONES COLECTIVAS (PIC); PARA LA PROMOCION DE LA SALUD Y PREVENCION DE LA ENFERMEDAD EN EL MARCO DE ATENCION PRIMARIA EN SALUD (AP\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4063175\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4292664\",\"nombre_proceso\":\"Prestar los servicios integrales como operador de comunicaciones para la preproducción; producción; postproducción y emisión de piezas; prog\",\"descripcion\":\"Prestar los servicios integrales como operador de comunicaciones para la preproducción; producción; postproducción y emisión de piezas; programas y campañas audiovisuales para la p\",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4209126\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4328089\",\"nombre_proceso\":\"ADQUISICIÓN DE LICENCIAMIENTO; NETAPP Y OPTIMIZACIÓN PARA LA PLATAFORMA DE RECEPCIÓN Y ANÁLISIS DE COMUNICACIONES INTERCEPTADAS\",\"descripcion\":\"ADQUISICIÓN DE LICENCIAMIENTO; NETAPP Y OPTIMIZACIÓN PARA LA PLATAFORMA DE RECEPCIÓN Y ANÁLISIS DE COMUNICACIONES INTERCEPTADAS\",\"entidad\":\"DIRECCIÓN DE INVESTIGACIÓN CRIMINAL E INTERPOL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4235876\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4374101\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS - COMPENSAR\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS - COMPENSAR\",\"entidad\":\"TRANSMILENIO S.A.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4279522\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4398956\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS - COMPENSAR\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS - COMPENSAR\",\"entidad\":\"TRANSMILENIO S.A.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4312142\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4442088\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN A LA GERENCIA DE LAS TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES DEL DISTRITO DE BARRAN\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN A LA GERENCIA DE LAS TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES DEL DISTRITO DE BARRANQUILLA EN EL SOPORTE; MANTENIMIENTO Y AC\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4345290\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4452101\",\"nombre_proceso\":\"GERENCIA INTEGRAL Y COORDINACION PARA LA ADMINISTRACION; MANTENIMIENTO Y REPARACION DE LA INSFRAESTRUCTURA FISICA DE LOS ESCENARIOS DEPORTIV\",\"descripcion\":\"GERENCIA INTEGRAL Y COORDINACION PARA LA ADMINISTRACION; MANTENIMIENTO Y REPARACION DE LA INSFRAESTRUCTURA FISICA DE LOS ESCENARIOS DEPORTIVOS DEL DISTRITO DE BARRANQUILLA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4356428\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4680510\",\"nombre_proceso\":\"Servicios en ejecución de las campañas de comunicación pública; en el marco del programa de información y comunicación pública\",\"descripcion\":\"Servicios en ejecución de las campañas de comunicación pública; en el marco del programa de información y comunicación pública\",\"entidad\":\"METRO DE BOGOTA SA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4582575\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4714843\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS PARA LA REALIZACIÓN DE ACTIVIDADES DE BIENESTAR PARA EL PERSONAL QUE INTEGRA LA SECCIONAL DE TRÁNSITO Y TRANSPORTE D\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS PARA LA REALIZACIÓN DE ACTIVIDADES DE BIENESTAR PARA EL PERSONAL QUE INTEGRA LA SECCIONAL DE TRÁNSITO Y TRANSPORTE DE BOGOTÁ\",\"entidad\":\"FONDO ROTATORIO DE LA POLICIA NACIONAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4618181\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4760867\",\"nombre_proceso\":\"CONVENIO DE ASOCIACION\",\"descripcion\":\"CONVENIO DE ASOCIACION\",\"entidad\":\"GOBERNACIÓN DE BOYACÁ\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4663504\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4885898\",\"nombre_proceso\":\"BOLSA MERCANTIL DE COLOMBIA S.A.\",\"descripcion\":\"BOLSA MERCANTIL DE COLOMBIA S.A.\",\"entidad\":\"MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4787508\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4888521\",\"nombre_proceso\":\"Aunar esfuerzos técnicos; administrativos y financieros para apoyar el desarrollo de las actividades de los programas de la Dirección de Fom\",\"descripcion\":\"Aunar esfuerzos técnicos; administrativos y financieros para apoyar el desarrollo de las actividades de los programas de la Dirección de Fomento y Desarrollo; enfocado a la promoci\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4791313\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4905303\",\"nombre_proceso\":\"Aunar esfuerzos técnicos; administrativos y financieros para apoyar el desarrollo de las actividades de los programas de la Dirección de Fom\",\"descripcion\":\"Aunar esfuerzos técnicos; administrativos y financieros para apoyar el desarrollo de las actividades de los programas de la Dirección de Fomento y Desarrollo; enfocado a la promoci\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4812141\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4932500\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"AGENCIA DE DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4833910\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5009790\",\"nombre_proceso\":\"Contrato Interadministrativo - ENTerritorio\",\"descripcion\":\"Contrato Interadministrativo - ENTerritorio\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4909374\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5077712\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA BRINDAR ACOMPAÑAMIENTO A LA SECRETARIA DISTRITAL DE RECREACION Y DEPORTE PARA EL DESARROL\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA BRINDAR ACOMPAÑAMIENTO A LA SECRETARIA DISTRITAL DE RECREACION Y DEPORTE PARA EL DESARROLLO DE LA ESTRATEGIA DE FOMENTO DEL DEPOR\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4976590\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5079345\",\"nombre_proceso\":\"SERVICIOS ESPECIALIZADOS DE GESTIÓN DOCUMENTAL\",\"descripcion\":\"SERVICIOS ESPECIALIZADOS DE GESTIÓN DOCUMENTAL\",\"entidad\":\"PROCURADURIA GENERAL DE LA NACION\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4978414\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5116390\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"MINISTERIO DE AMBIENTE Y DESARROLLO SOSTENIBLE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5015617\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5128143\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"MINISTERIO DE AMBIENTE Y DESARROLLO SOSTENIBLE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5026468\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5134421\",\"nombre_proceso\":\"DESARROLLAR EL PROCESO DE EVALUACIÓN INTEGRAL DE LOS PROYECTOS QUE PARTICIPEN EN LAS CONVOCATORIAS DEL PROGRAMA NACIONAL DE CONCERTACIÓN CUL\",\"descripcion\":\"DESARROLLAR EL PROCESO DE EVALUACIÓN INTEGRAL DE LOS PROYECTOS QUE PARTICIPEN EN LAS CONVOCATORIAS DEL PROGRAMA NACIONAL DE CONCERTACIÓN CULTURAL 2024\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5032868\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5161265\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN DE LA SECRETARÍA DISTRITAL DE RECREACIÓN Y DEPORTE PARA EL DESARROLLO DEL PROGRAMA BARRANQUILL\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN DE LA SECRETARÍA DISTRITAL DE RECREACIÓN Y DEPORTE PARA EL DESARROLLO DEL PROGRAMA BARRANQUILLA ACTIVA Y RECREATIVA A TRAVES DE LA EST\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5059181\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5269290\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO\",\"entidad\":\"GOBERNACION DE NARIÑO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5166271\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5291411\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5248108\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5319517\",\"nombre_proceso\":\"Aunar esfuerzos técnicos y recursos físicos para la  implementación de la ruta o metodología que permita a través  del diálogo e intercambio\",\"descripcion\":\"Aunar esfuerzos técnicos y recursos físicos para la  implementación de la ruta o metodología que permita a través  del diálogo e intercambio de saberes; adelantar las acciones  nec\",\"entidad\":\"MINISTERIO DE MINAS Y ENERGIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5215643\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5327151\",\"nombre_proceso\":\"UNION TEMPORAL RH\",\"descripcion\":\"UNION TEMPORAL RH\",\"entidad\":\"DEPARTAMENTO ADMINISTRATIVO DE LA PRESIDENCIA DE LA REPÚBLICA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5223079\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5408655\",\"nombre_proceso\":\"ARRENDAMIENTO INMUEBLE SEDE PRINCIPAL BOGOTÁ\",\"descripcion\":\"ARRENDAMIENTO INMUEBLE SEDE PRINCIPAL BOGOTÁ\",\"entidad\":\"SUPERINTENDENCIA NACIONAL DE SALUD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5311225\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5463625\",\"nombre_proceso\":\"PROCOMERCIO S.A\",\"descripcion\":\"PROCOMERCIO S.A\",\"entidad\":\"MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5360131\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5935970\",\"nombre_proceso\":\"Prestación de servicios profesionales para brindar asesoría y seguimiento de la estrategia  de fortalecimiento de capacidad técnica para el \",\"descripcion\":\"Prestación de servicios profesionales para brindar asesoría y seguimiento de la estrategia  de fortalecimiento de capacidad técnica para el ejercicio misional preventivo en la  imp\",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5929808\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5943339\",\"nombre_proceso\":\"CONTRATAR EL SERVICIO Y SOSTENIBILIDAD DE LA SOLUCIÓN TECNOLÓGICA INTEGRADA  TIPO SaaS DENOMINADA PARA LA DEFENSORÍA DEL PUEBLO APP CONTIGO;\",\"descripcion\":\"CONTRATAR EL SERVICIO Y SOSTENIBILIDAD DE LA SOLUCIÓN TECNOLÓGICA INTEGRADA  TIPO SaaS DENOMINADA PARA LA DEFENSORÍA DEL PUEBLO APP CONTIGO; PARA LA  TOMA DE DECISIONES OPORTUNAS E\",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5854184\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5987887\",\"nombre_proceso\":\"Fondo Mixto de Promoción de la Cultura y las Artes de Boyacá\",\"descripcion\":\"Fondo Mixto de Promoción de la Cultura y las Artes de Boyacá\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5876910\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6116403\",\"nombre_proceso\":\"ALIANZA PUBLICA PARA EL DESDARROLLO INTEGRAL (ALDESARROLLO)\",\"descripcion\":\"ALIANZA PUBLICA PARA EL DESDARROLLO INTEGRAL (ALDESARROLLO)\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6005101\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6237418\",\"nombre_proceso\":\"Universidad Nacional de Colombia\",\"descripcion\":\"Universidad Nacional de Colombia\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6128253\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6477624\",\"nombre_proceso\":\"INTERNEXA S.A.- ACUERDO ESPECÍFICO No. 4\",\"descripcion\":\"INTERNEXA S.A.- ACUERDO ESPECÍFICO No. 4\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6363642\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6487930\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA CUSTODIAR; ALMACENAR Y ADMINISTRAR ARCHIVOS FISICOS DE LA ALCALDIA DISTRITAL DE BARRANQUI\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA CUSTODIAR; ALMACENAR Y ADMINISTRAR ARCHIVOS FISICOS DE LA ALCALDIA DISTRITAL DE BARRANQUILLA; ASI COMO TAMBIEN REGISTRAR; ATENDER\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6373487\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6509936\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA CUSTODIAR; ALMACENAR Y ADMINISTRAR ARCHIVOS FISICOS DE LA ALCALDIA DISTRITAL DE BARRANQUI\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA CUSTODIAR; ALMACENAR Y ADMINISTRAR ARCHIVOS FISICOS DE LA ALCALDIA DISTRITAL DE BARRANQUILLA; ASI COMO TAMBIEN REGISTRAR; ATENDER\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6395315\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6527293\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO FDL FONTIBON\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO FDL FONTIBON\",\"entidad\":\"(Secretaría Distrital de Integración Social)\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6413656\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6594810\",\"nombre_proceso\":\"CONVENIO UNIVERSIDAD DE CALDAS\",\"descripcion\":\"CONVENIO UNIVERSIDAD DE CALDAS\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6482253\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6601074\",\"nombre_proceso\":\"CI-011-2024\",\"descripcion\":\"CI-011-2024\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6484864\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6635424\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA ACOMPAÑAR A LA ENTIDAD EN EL DESARROLLO DE LAS PRESENTACIONES ARTÍSTICAS; CULTURALES; INC\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA ACOMPAÑAR A LA ENTIDAD EN EL DESARROLLO DE LAS PRESENTACIONES ARTÍSTICAS; CULTURALES; INCLUIDO EL CONCIERTO DE MÚSICA GOSPEL; ENM\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6520474\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.666327\",\"nombre_proceso\":\"SERVICIOS POSTALES Y GESTIÓN DOCUMENTAL\",\"descripcion\":\"SERVICIOS POSTALES Y GESTIÓN DOCUMENTAL\",\"entidad\":\"DEPARTAMENTO NACIONAL DE PLANEACIÓN\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.638702\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6735811\",\"nombre_proceso\":\"CONVENIO CENSO HABITANTE DE CALLE SDIS-SDP\",\"descripcion\":\"CONVENIO CENSO HABITANTE DE CALLE SDIS-SDP\",\"entidad\":\"(Secretaría Distrital de Integración Social)\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6619354\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6808401\",\"nombre_proceso\":\"FIDUCOLDEX-FONDO MUJER LIBRE Y REPRODUCTIVA\",\"descripcion\":\"FIDUCOLDEX-FONDO MUJER LIBRE Y REPRODUCTIVA\",\"entidad\":\"MINISTERIO DE IGUALDAD Y EQUIDAD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6706476\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6824136\",\"nombre_proceso\":\"DEPARTAMENTO DE VAUPÉS\",\"descripcion\":\"DEPARTAMENTO DE VAUPÉS\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6706088\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6827516\",\"nombre_proceso\":\"FIDUCOLDEX-FONDO MUJER LIBRE Y PRODUCTIVA\",\"descripcion\":\"FIDUCOLDEX-FONDO MUJER LIBRE Y PRODUCTIVA\",\"entidad\":\"MINISTERIO DE IGUALDAD Y EQUIDAD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6709881\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6835303\",\"nombre_proceso\":\"CONTRATO UNAL CONCERTACIONES\",\"descripcion\":\"CONTRATO UNAL CONCERTACIONES\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6717498\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6995311\",\"nombre_proceso\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA FORTALECIMIENTO DE LA INFRAESTRUCTURA TECNOLÓGICA Y DINAMIZACIÓN DE LA IMPLEMEN\",\"descripcion\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA FORTALECIMIENTO DE LA INFRAESTRUCTURA TECNOLÓGICA Y DINAMIZACIÓN DE LA IMPLEMENTACIÓN DE LA POLÍTICA DE GOBIERNO DIGITA\",\"entidad\":\"GOBERNACION DE SUCRE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6878334\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7278319\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO CON LA EMPRESA GENERADORA DE ENERGIA DEL TOLIMA EGETSA SA ESP\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO CON LA EMPRESA GENERADORA DE ENERGIA DEL TOLIMA EGETSA SA ESP\",\"entidad\":\"GOBIERNO DEPARTAMENTAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7166709\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7280635\",\"nombre_proceso\":\"CORPORACION FINANCIERA INTERNACIONAL\",\"descripcion\":\"CORPORACION FINANCIERA INTERNACIONAL\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7338894\",\"nombre_proceso\":\"Contratar bajo la modalidad de arrendamiento el inmueble; para el funcionamiento de la sede Archivo de Gestión; para la administración; cons\",\"descripcion\":\"Contratar bajo la modalidad de arrendamiento el inmueble; para el funcionamiento de la sede Archivo de Gestión; para la administración; conservación y custodia de la información de\",\"entidad\":\"PROSPERIDAD SOCIAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7222199\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7431031\",\"nombre_proceso\":\"EJECUTAR ACTIVIDADES ADMINISTRATIVAS; OPERATIVAS; TÉCNICAS; ASISTENCIALES Y LOGÍSTICAS QUE FORTALEZCAN LA GESTIÓN Y EL FUNCIONAMIENTO DE LAS\",\"descripcion\":\"EJECUTAR ACTIVIDADES ADMINISTRATIVAS; OPERATIVAS; TÉCNICAS; ASISTENCIALES Y LOGÍSTICAS QUE FORTALEZCAN LA GESTIÓN Y EL FUNCIONAMIENTO DE LAS 24 INSTITUCIONES EDUCATIVAS OFICIALES Y\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7321180\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7449230\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA SOPORTAR AL MUNICIPIO DE ITAGÜÍ EN LA EJECUCIÓN DE ACTIVIDADES OPERATIVAS; LOGÍSTICAS; TÉ\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA SOPORTAR AL MUNICIPIO DE ITAGÜÍ EN LA EJECUCIÓN DE ACTIVIDADES OPERATIVAS; LOGÍSTICAS; TÉCNICAS; ASISTENCIALES Y ADMINISTRATIVAS \",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7330359\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7466190\",\"nombre_proceso\":\"BRINDAR ATENCION BASICA INTEGRAL Y PROTECCION AL ADULTO MAYOR; EN LA MODALIDAD INTERNADO; PARA GARANTIZAR EL MEJORAMIENTO DE LA CALIDAD DE V\",\"descripcion\":\"BRINDAR ATENCION BASICA INTEGRAL Y PROTECCION AL ADULTO MAYOR; EN LA MODALIDAD INTERNADO; PARA GARANTIZAR EL MEJORAMIENTO DE LA CALIDAD DE VIDA DE LOS ADULTOS MAYORES VULNERABLES Y\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7346489\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.755348\",\"nombre_proceso\":\"Prestación de servicios profesionales especializados y de apoyo a la gestión para realizar la evaluación en fase de precalificación del proc\",\"descripcion\":\"Prestación de servicios profesionales especializados y de apoyo a la gestión para realizar la evaluación en fase de precalificación del proceso GT-LPI-001-2018\",\"entidad\":\"METRO DE BOGOTA SA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.727638\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7561573\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO PARA EL ARRENDAMIENTO DE VEHÍCULOS PARA APOYAR A LA FUERZA PÚBLICA; ORGANISMOS DE SEGURIDAD Y OTRAS ENTIDADES D\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO PARA EL ARRENDAMIENTO DE VEHÍCULOS PARA APOYAR A LA FUERZA PÚBLICA; ORGANISMOS DE SEGURIDAD Y OTRAS ENTIDADES DEL MUNICIPIO DE RIONEGRO - ANTIOQUIA\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7441197\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7578958\",\"nombre_proceso\":\"IMPLEMENTACIÓN DE ACCIONES PARA EL FORTALECIMIENTO DE LA GESTIÓN CATASTRAL DEL ÁREA METROPOLITANA DEL VALLE DE ABURRÁ FASE 2025\",\"descripcion\":\"IMPLEMENTACIÓN DE ACCIONES PARA EL FORTALECIMIENTO DE LA GESTIÓN CATASTRAL DEL ÁREA METROPOLITANA DEL VALLE DE ABURRÁ FASE 2025\",\"entidad\":\"AREA METROPOLITANA DEL VALLE DE ABURRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7489575\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7603157\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO - UNP\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO - UNP\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7483281\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.769107\",\"nombre_proceso\":\"FENASCOL\",\"descripcion\":\"FENASCOL\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.737357\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7700801\",\"nombre_proceso\":\"ORGANIZACIÓN WIWA YUGUMAIUN BUNKUANARRUA TAIRONA\",\"descripcion\":\"ORGANIZACIÓN WIWA YUGUMAIUN BUNKUANARRUA TAIRONA\",\"entidad\":\"GOBERNACIÓN DE LA GUAJIRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7579096\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7710982\",\"nombre_proceso\":\"PRESTACION DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACION ETNICA DE LA INSTITUCION ETNOEDUCATIVA INTEGRAL RURAL JACOBO ARE\",\"descripcion\":\"PRESTACION DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACION ETNICA DE LA INSTITUCION ETNOEDUCATIVA INTEGRAL RURAL JACOBO ARENDS GOURIYU; INSTITUCION ETNOEDUCATIVA I\",\"entidad\":\"MUNICIPIO DE URIBIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7589624\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7769056\",\"nombre_proceso\":\"CONSEJO DE AUTORIDADES  TRADICIONALES INDÍGENAS DEL CHOCÓ_ CATICH\",\"descripcion\":\"CONSEJO DE AUTORIDADES  TRADICIONALES INDÍGENAS DEL CHOCÓ_ CATICH\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7651491\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7844264\",\"nombre_proceso\":\"EMPRESA DE TELECOMUNICACIONES DE BOGOTÁ ETB S.A. ESP\",\"descripcion\":\"EMPRESA DE TELECOMUNICACIONES DE BOGOTÁ ETB S.A. ESP\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7865778\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7744188\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7961767\",\"nombre_proceso\":\"ESTUDIOS PARA EL ACOTAMIENTO DE LA RONDA HÍDRICA DEL RÍO MEDELLÍN - ABURRÁ EN EL VALLE DE ABURRÁ\",\"descripcion\":\"ESTUDIOS PARA EL ACOTAMIENTO DE LA RONDA HÍDRICA DEL RÍO MEDELLÍN - ABURRÁ EN EL VALLE DE ABURRÁ\",\"entidad\":\"AREA METROPOLITANA DEL VALLE DE ABURRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7977449\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7966756\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA  POLÍTICA DE EST\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA  POLÍTICA DE ESTADO PARA EL DESARROLLO INTEGRAL DE LA PR\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7843892\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8005315\",\"nombre_proceso\":\"CONTRATAR EL ARRENDAMIENTO DEL ESPACIO FÍSICO PARA LA CONSERVACIÓN DE LOS ARCHIVOS DE LA GESTIÓN DOCUMENTAL DE LA SECRETARÍA DISTRITAL DE MO\",\"descripcion\":\"CONTRATAR EL ARRENDAMIENTO DEL ESPACIO FÍSICO PARA LA CONSERVACIÓN DE LOS ARCHIVOS DE LA GESTIÓN DOCUMENTAL DE LA SECRETARÍA DISTRITAL DE MOVILIDAD.\",\"entidad\":\"SECRETARÍA DISTRITAL DE MOVILIDAD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7889991\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8018431\",\"nombre_proceso\":\"LA ADQUISICIÓN BLINDAJES AERONÁUTICOS; SEGÚN ANEXO TÉCNICO; COMO SOPORTE PARA LA FUNCIONALIDAD Y OPERATIVIDAD DE LOS SISTEMAS DE ARMAS DE LA\",\"descripcion\":\"LA ADQUISICIÓN BLINDAJES AERONÁUTICOS; SEGÚN ANEXO TÉCNICO; COMO SOPORTE PARA LA FUNCIONALIDAD Y OPERATIVIDAD DE LOS SISTEMAS DE ARMAS DE LA FUERZA AEROESPACIAL COLOMBIANA.\",\"entidad\":\"FUERZA AEROESPACIAL COLOMBIANA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7912562\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8020511\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO DE MANDATO SIN REPRESENTACION PARA LA ADMNISTRACION DE LOS SERVICIOS DE VIGILANCIA Y SEGURIDAD PRIVADA DEL PERS\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO DE MANDATO SIN REPRESENTACION PARA LA ADMNISTRACION DE LOS SERVICIOS DE VIGILANCIA Y SEGURIDAD PRIVADA DEL PERSONAL LOS BIENES INMUEBLES Y SU CONTENIDO\",\"entidad\":\"AREA METROPOLITANA DEL VALLE DE ABURRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7912558\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8035689\",\"nombre_proceso\":\"ASESOFTWARE S.A.S\",\"descripcion\":\"ASESOFTWARE S.A.S\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7921422\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8037736\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7918350\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8057757\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"GOBERNACIÓN DE BOYACÁ\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7960849\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8073472\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN DE  LA OFICINA PARA LA SEGURIDAD Y CONVIVENCIA CIUDADANA DEL DISTRITO DE BARRANQUILLA\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN DE  LA OFICINA PARA LA SEGURIDAD Y CONVIVENCIA CIUDADANA DEL DISTRITO DE BARRANQUILLA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7951692\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8112173\",\"nombre_proceso\":\"CONTRATO EMPRESTITO DE CREDITO DE TESORERÍA A CORTO PLAZO CELEBRADO ENTRE EL DISTRITO ESPECIAL; INDUSTRIAL Y PORTUARIO DE BARRANQUILLA Y ALI\",\"descripcion\":\"CONTRATO EMPRESTITO DE CREDITO DE TESORERÍA A CORTO PLAZO CELEBRADO ENTRE EL DISTRITO ESPECIAL; INDUSTRIAL Y PORTUARIO DE BARRANQUILLA Y ALIANZA FIDUCIARIA S.A.\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7990225\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8160797\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA  GESTIÓN PARA LA IMPLEMENTACIÓN DE LA ESTRATEGIA SISTEMAS LOCALES DE JUSTICIA  POR PARTE DE LA SECRETA\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA  GESTIÓN PARA LA IMPLEMENTACIÓN DE LA ESTRATEGIA SISTEMAS LOCALES DE JUSTICIA  POR PARTE DE LA SECRETARÍA DE GOBIERNO DEL DISTRITO DE BARRANQU\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8036751\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8184970\",\"nombre_proceso\":\"CI-0866-2025\",\"descripcion\":\"CI-0866-2025\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8060041\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8257854\",\"nombre_proceso\":\"CONVENIO ALCANTARILLADO ISTMINA\",\"descripcion\":\"CONVENIO ALCANTARILLADO ISTMINA\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8132711\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8326782\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8200590\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8326892\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO PARA LA GERENCIA Y OPERACIÓN INTEGRAL DEL COMPONENTE RIOBUS INTEGRADO AL ESPACIO PUBLICO DENOMINADO GRAN MALECO\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO PARA LA GERENCIA Y OPERACIÓN INTEGRAL DEL COMPONENTE RIOBUS INTEGRADO AL ESPACIO PUBLICO DENOMINADO GRAN MALECON DEL RIO\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8200728\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8376652\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO PARA LOS ESTUDIOS Y DISEÑOS TÉCNICOS INTEGRALES DE LA NUEVA PLAZA DE MERCADO DEL MUNICIPIO DE RIONEGRO ANTIOQUI\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO PARA LOS ESTUDIOS Y DISEÑOS TÉCNICOS INTEGRALES DE LA NUEVA PLAZA DE MERCADO DEL MUNICIPIO DE RIONEGRO ANTIOQUIA; ASÍ COMO LA FORMULACIÓN Y ESTRUCTURAC\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8250335\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8686546\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS PROFESIONALES PARA  BRINDAR ACOMPAÑAMIENTO A LA SECRETARÍA DISTRITAL DE GESTIÓN HUMANA  EN EL PLAN INSTITUCIONAL DE \",\"descripcion\":\"PRESTACIÓN DE SERVICIOS PROFESIONALES PARA  BRINDAR ACOMPAÑAMIENTO A LA SECRETARÍA DISTRITAL DE GESTIÓN HUMANA  EN EL PLAN INSTITUCIONAL DE CAPACITACIÓN DIRIGIDO A LOS FUNCIONARIOS\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8554272\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8691802\",\"nombre_proceso\":\"LA ASOCIACIÓN DE AUTORIDADES TRADICIONALES AWA - ORGANIZACIÓN UNIDAD INDÍGENA DEL PUEBLO AWA- UNIPA\",\"descripcion\":\"LA ASOCIACIÓN DE AUTORIDADES TRADICIONALES AWA - ORGANIZACIÓN UNIDAD INDÍGENA DEL PUEBLO AWA- UNIPA\",\"entidad\":\"MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8559767\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8801252\",\"nombre_proceso\":\"CONVENIO SAN ANDRES\",\"descripcion\":\"CONVENIO SAN ANDRES\",\"entidad\":\"AGENCIA DE DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8672321\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8855449\",\"nombre_proceso\":\"DEPARTAMENTO DE PUTUMAYO\",\"descripcion\":\"DEPARTAMENTO DE PUTUMAYO\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8725864\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8934765\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8800669\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8985249\",\"nombre_proceso\":\"Prestar los servicios de organización de fondo documental acumulado; digitalización de documentos y administración; almacenamiento y custodi\",\"descripcion\":\"Prestar los servicios de organización de fondo documental acumulado; digitalización de documentos y administración; almacenamiento y custodia del archivo del CNE; en cumplimiento d\",\"entidad\":\"Consejo Nacional Electoral\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8851305\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8999605\",\"nombre_proceso\":\"AGENCIA NACIONAL INMOBILIARIA VIRGILIO BARCO VARGAS\",\"descripcion\":\"AGENCIA NACIONAL INMOBILIARIA VIRGILIO BARCO VARGAS\",\"entidad\":\"MINISTERIO DE HACIENDA Y CREDITO PUBLICO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8864269\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9018983\",\"nombre_proceso\":\"CONECTIVIDAD ESCOLAR 2025\",\"descripcion\":\"CONECTIVIDAD ESCOLAR 2025\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8884094\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9021155\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO POR ADMINISTRACIÓN DELEGADA DE RECURSOS PARA LA CONSTRUCCIÓN E INTERVENTORIA DEL PROYECTO CIUDADELA EDUCATIVA 4\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO POR ADMINISTRACIÓN DELEGADA DE RECURSOS PARA LA CONSTRUCCIÓN E INTERVENTORIA DEL PROYECTO CIUDADELA EDUCATIVA 4.0 ETAPA 1 EN EL MUNICIPIO DE RIONEGRO.\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8886200\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9100088\",\"nombre_proceso\":\"CONVENIO GUAJIRA\",\"descripcion\":\"CONVENIO GUAJIRA\",\"entidad\":\"AGENCIA DE DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8967705\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9104218\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO - CABILDO KOGUI - DAI\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO - CABILDO KOGUI - DAI\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9120928\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARÍA DISTRITAL DE RECREACIÓN Y DEPORTES\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARÍA DISTRITAL DE RECREACIÓN Y DEPORTES\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8987221\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9130747\",\"nombre_proceso\":\"CONVENIO ARIT IM-169 Y IM-170\",\"descripcion\":\"CONVENIO ARIT IM-169 Y IM-170\",\"entidad\":\"MINISTERIO DE IGUALDAD Y EQUIDAD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8998068\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9145065\",\"nombre_proceso\":\"CONTRATACION DIRECTA - CCTV YOPAL CASANARE - SPS\",\"descripcion\":\"CONTRATACION DIRECTA - CCTV YOPAL CASANARE - SPS\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9006742\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9171487\",\"nombre_proceso\":\"FNV-0025-2025\",\"descripcion\":\"FNV-0025-2025\",\"entidad\":\"FONDO NACIONAL DE VIVIENDA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9033424\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9217348\",\"nombre_proceso\":\"CONVENIO ASOMURA\",\"descripcion\":\"CONVENIO ASOMURA\",\"entidad\":\"AGENCIA DE DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9086653\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9224040\",\"nombre_proceso\":\"MEJORAMIENTO DE LA INFRAESTRUCTURA EDUCATIVA\",\"descripcion\":\"MEJORAMIENTO DE LA INFRAESTRUCTURA EDUCATIVA\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9084618\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9224503\",\"nombre_proceso\":\"GOBERNACIÓN HUILA\",\"descripcion\":\"GOBERNACIÓN HUILA\",\"entidad\":\"COMPUTADORES PARA EDUCAR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9084859\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9229871\",\"nombre_proceso\":\"AUNAR ESFUERZOS TECNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA LA EJECUCION DEL PROYECTO DENOMINADO CONSTRUCCIÓN DE UN CENTRO DE ALTO RENDIMIE\",\"descripcion\":\"AUNAR ESFUERZOS TECNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA LA EJECUCION DEL PROYECTO DENOMINADO CONSTRUCCIÓN DE UN CENTRO DE ALTO RENDIMIENTO DEPORTIVO EN LA CIUDAD DE MONTERÍA; \",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9092192\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9231569\",\"nombre_proceso\":\"AUNAR ESFUERZOS TECNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA LA EJECUCION DEL PROYECTO DENOMINADO  CONSTRUCCIÓN DE LA VILLA OLIMPICA FASE II\",\"descripcion\":\"AUNAR ESFUERZOS TECNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA LA EJECUCION DEL PROYECTO DENOMINADO  CONSTRUCCIÓN DE LA VILLA OLIMPICA FASE II; COLISEO DE PELOTAS MUNICIPIO DE TOLÚ; \",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9092190\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9241289\",\"nombre_proceso\":\"CONVENIO DE ASOCIACIÓN\",\"descripcion\":\"CONVENIO DE ASOCIACIÓN\",\"entidad\":\"GOBIERNO DEPARTAMENTAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9101774\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9311175\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9185835\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9449488\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS PROFESIONALES PARA EL ACOMPAÑAMIENTO; ASESORÍA; APOYO Y SOPORTE A LA GESTIÓN; ASÍ COMO A LA ADMINISTRACIÓN DEL DATA \",\"descripcion\":\"PRESTACIÓN DE SERVICIOS PROFESIONALES PARA EL ACOMPAÑAMIENTO; ASESORÍA; APOYO Y SOPORTE A LA GESTIÓN; ASÍ COMO A LA ADMINISTRACIÓN DEL DATA CENTER; SERVICIOS DIGITALES; SISTEMAS DE\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9317035\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9467668\",\"nombre_proceso\":\"DE CERO A SIEMPRE\",\"descripcion\":\"DE CERO A SIEMPRE\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9323821\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9495453\",\"nombre_proceso\":\"CONVENIO DE COOPERACION INTERNACIONAL - COMITATO INTERNAZIONALE\",\"descripcion\":\"CONVENIO DE COOPERACION INTERNACIONAL - COMITATO INTERNAZIONALE\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9347839\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9516203\",\"nombre_proceso\":\"APOYAR A LA SECRETARÍA DE HACIENDA DEL MUNICIPIO DE ITAGÜÍ EN EL PROCESO CONTABLE; DE TESORERÍA; PENSIONAL; DE SEGURIDAD SOCIAL Y ADMINISTRA\",\"descripcion\":\"APOYAR A LA SECRETARÍA DE HACIENDA DEL MUNICIPIO DE ITAGÜÍ EN EL PROCESO CONTABLE; DE TESORERÍA; PENSIONAL; DE SEGURIDAD SOCIAL Y ADMINISTRATIVOS; ORIENTADOS AL SANEAMIENTO FINANCI\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9366959\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9517307\",\"nombre_proceso\":\"PRESTACIÒN DE SERVICIOS PROFESIONALES\",\"descripcion\":\"PRESTACIÒN DE SERVICIOS PROFESIONALES\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9367621\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9890719\",\"nombre_proceso\":\"ARRENDAMIENTO DE BIEN INMUEBLE PARA EL FUNCIONAMIENTO DE LAS DEPENDENCIAS DE LA CONTRALORIA DISTRITAL DE BARRANQUILLA EN EL MARCO DEL CONVEN\",\"descripcion\":\"ARRENDAMIENTO DE BIEN INMUEBLE PARA EL FUNCIONAMIENTO DE LAS DEPENDENCIAS DE LA CONTRALORIA DISTRITAL DE BARRANQUILLA EN EL MARCO DEL CONVENIO INTERADMINISTRATIVO No. CD-48-2025-84\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9739742\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9893329\",\"nombre_proceso\":\"ASOCAITADO- AYATALII\",\"descripcion\":\"ASOCAITADO- AYATALII\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9819678\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9897571\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA DESARROLLAR LA ESTRATEGIA DE PROMOCIÓN DE LA SALUD MENTAL MIS EMOCIONES; MI MAYOR RIQUEZA\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA DESARROLLAR LA ESTRATEGIA DE PROMOCIÓN DE LA SALUD MENTAL MIS EMOCIONES; MI MAYOR RIQUEZA EN EL DISTRITO DE BARRANQUILLA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9757193\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9912169\",\"nombre_proceso\":\"Centros de Vida promoción de la salud\",\"descripcion\":\"Centros de Vida promoción de la salud\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9762336\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9917266\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARIA DISTRITAL DE RECREACIÓN Y DEPORTES PARA EL DESARROLLO DE RETOS ORIENTADOS AL A\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARIA DISTRITAL DE RECREACIÓN Y DEPORTES PARA EL DESARROLLO DE RETOS ORIENTADOS AL APROVECHAMIENTO DEL TIEMPO LIBRE MEDIANTE\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9767946\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9927170\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA EL FORTALECIMIENTO DE ESTRATEGIAS DE PEDAGOGÍA CIUDADANA ORIENTADAS A LA DEMOCRACIA PARTI\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA EL FORTALECIMIENTO DE ESTRATEGIAS DE PEDAGOGÍA CIUDADANA ORIENTADAS A LA DEMOCRACIA PARTICIPATIVA Y PLURALISTA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9777475\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9958123\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9811133\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9963832\",\"nombre_proceso\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA EDUCATIVA DE LA POBLACIÓN ÉTNICA DE LOS CENTROS ETNOEDUCATIVOS UTPURAI- UBICADO EN LA ZO\",\"descripcion\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA EDUCATIVA DE LA POBLACIÓN ÉTNICA DE LOS CENTROS ETNOEDUCATIVOS UTPURAI- UBICADO EN LA ZONA RURAL ÉTNICA DEL MUNICIPIO DE ALBANIA\",\"entidad\":\"GOBERNACIÓN DE LA GUAJIRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9824442\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9991057\",\"nombre_proceso\":\"CANAL REGIONAL DE TELEVISIÓN TEVEANDINA S.A.S. - TEVEANDINA S.A.S.\",\"descripcion\":\"CANAL REGIONAL DE TELEVISIÓN TEVEANDINA S.A.S. - TEVEANDINA S.A.S.\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9843067\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9993489\",\"nombre_proceso\":\"CORPORACIÓN COLOMBIA CREA TALENTO - COCREA\",\"descripcion\":\"CORPORACIÓN COLOMBIA CREA TALENTO - COCREA\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9844553\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"}]")
FUENTE = "Datos/tc2_corpus_2026.json"
BLOB_FUENTE = "4b1ac0dd22137984b7e81864a70e04075af2af39"
ESQUEMA = ["id_proceso","nombre_proceso","descripcion","entidad","url_secop","fuente","tipo_registro"]
assert len(CORPUS)==200 and all(set(r)==set(ESQUEMA) for r in CORPUS)
assert len({r["id_proceso"] for r in CORPUS})==200
HASH_SNAPSHOT = sha(canon(CORPUS))
print("Snapshot",len(CORPUS),"procesos, SHA-256 canónico",HASH_SNAPSHOT)


In [ ]:
df=pd.DataFrame(CORPUS)
assert df.id_proceso.is_unique
display(df[["id_proceso","entidad","nombre_proceso"]].head(5))
conteo=df.groupby("entidad",dropna=False).size()
assert conteo.sum()==len(df)
display(conteo.sort_values(ascending=False).head(10))
copia=pd.concat([df.copy(),df.head(5)],ignore_index=True)
copia.loc[0,"entidad"]=""
assert len(copia)==205 and copia.id_proceso.nunique()==200
print({"filas":len(copia),"unicos":copia.id_proceso.nunique(),"repeticiones":int(copia.id_proceso.duplicated().sum()),"entidad_vacia":int(copia.entidad.eq("").sum())})
assert sha(canon(CORPUS))==HASH_SNAPSHOT
def tokens(s):
    norm=unicodedata.normalize("NFD",str(s).casefold())
    return re.findall(r"\w+", "".join(c for c in norm if not unicodedata.combining(c)))
termino="salud"
listas=[tokens(r["descripcion"]) for r in CORPUS]
print("Término",termino,"ocurrencias",sum(t.count(termino) for t in listas),"procesos",sum(termino in t for t in listas))

N=80  # modifica a 40 o 120; mismo orden que presentación
assert 1<=N<=len(df)
muestra=df.iloc[:N].copy()
agrupada=muestra.groupby("entidad",dropna=False).size()
assert int(agrupada.sum())==N and muestra.id_proceso.is_unique
print("Selección",N,"entidades",len(agrupada)); display(agrupada.sort_values(ascending=False).head(7))


## A4 · HTTP, errores y consulta optativa
Una petición válida no garantiza cobertura. La función clasifica respuestas de ejemplo: 200 requiere validar JSON y esquema; 429/503 pueden requerir reintento acotado; 401/403 exigen revisar acceso; timeout no equivale a vacío. Si se omite una página fallida, el proceso no puede afirmar completitud. Un servidor real puede tener reglas adicionales. En el núcleo no usamos red.

La ruta opcional consulta cinco registros contractuales de la API pública. Se mantiene separada del corpus de procesos y no pretende demostrar paginación del universo. Si falla, se informa el error y no se fabrica una respuesta. No solicita credenciales. Actívala solo si quieres contrastar el servicio; el resto del cuaderno no depende de esta consulta.


In [ ]:
def estado_http(status):
    if status==200: return "validar cuerpo y esquema"
    if status in (429,503,"timeout"): return "reintento acotado según política"
    if status in (401,403): return "revisar acceso"
    return "registrar incidencia"
for status in (200,429,503,403,"timeout",404):
    print(status,estado_http(status))
assert estado_http("timeout")!="validar cuerpo y esquema"
CONSULTA_VIVA=False
if CONSULTA_VIVA:
    import requests
    url="https://www.datos.gov.co/resource/rpmr-utcd.json"
    params={"$limit":5,"$select":"nombre_de_la_entidad,departamento_entidad,valor_contrato"}
    try:
        r=requests.get(url,params=params,timeout=15)
        r.raise_for_status()
        registros=r.json()
        assert isinstance(registros,list) and len(registros)<=5
        print({"modo":"consulta viva","grano":"registro contractual","fuente":url,"filas":len(registros)})
    except (requests.RequestException,ValueError,AssertionError) as exc:
        print("Consulta viva no comprobada:",type(exc).__name__)
else:
    print("Consulta viva no ejecutada; snapshot disponible.")


## A5 · Paginación, orden y cobertura
Offset omite posiciones; limit fija máximo de filas por página. Para un universo congelado de N registros, el salto igual a limit y techo(N/limit) peticiones cubren todas las posiciones. Si el salto es menor, hay solapamiento; si es mayor, huecos. Una lista mutable puede desplazar registros aun con parámetros correctos. Para una fuente viva, el orden total y un mecanismo de versión o reconciliación deben establecerse según el servicio.

**Predice:** cuatro páginas de 50 con salto 40 devuelven 200 filas, pero ¿cuántas identidades? Ejecuta el control. Compara ausencias, extras y repeticiones, no solo conteo. Luego usa salto 50, cambia el tamaño a 37 y calcula automáticamente el número de peticiones. No avances si los invariantes fallan.

**Cómo se lee:** ventanas, IDs únicos y faltantes. **Qué nos dice:** cobertura exacta frente a este universo congelado. **Qué no permite concluir:** cobertura de un portal que cambia. **Error común:** usar HTTP 200 o número bruto de filas como prueba de adquisición completa.

En este laboratorio el orden total es la posición en el snapshot versionado; no reordenamos los registros antes de seleccionar un prefijo. En una consulta viva usa un orden total según el contrato de esa fuente, pero no confundas el orden con una instantánea.


In [ ]:
def adquirir(base,limite,salto,paginas):
    assert limite>0 and salto>0 and paginas>0
    esperado={r["id_proceso"] for r in base}
    assert len(esperado)==len(base)
    filas=[]; traza=[]
    for p in range(paginas):
        offset=p*salto; lote=base[offset:offset+limite]
        filas.extend(lote)
        traza.append({"pagina":p+1,"offset":offset,"filas":len(lote)})
    c=Counter(r["id_proceso"] for r in filas)
    control={"filas":len(filas),"unicos":len(c),"duplicados":sum(v-1 for v in c.values()),
             "ausentes":sorted(esperado-set(c)),"extras":sorted(set(c)-esperado)}
    return filas,control,traza
base=list(CORPUS)  # orden estable del snapshot versionado, igual al explorador
_,mal,_=adquirir(base,50,40,4)
assert mal["filas"]==200 and mal["unicos"]==170 and mal["duplicados"]==len(mal["ausentes"])==30
print("Salto incorrecto", {k:(len(v) if isinstance(v,list) else v) for k,v in mal.items()})
LIMIT=37
raw,control,traza=adquirir(base,LIMIT,LIMIT,math.ceil(len(base)/LIMIT))
assert control["unicos"]==200 and control["duplicados"]==0 and not control["ausentes"] and not control["extras"]
display(pd.DataFrame(traza))
print("Cobertura correcta",control)

# Contraejemplo de fuente mutable (no SECOP en vivo).
original=[f"P{i:02d}" for i in range(1,9)]
p1=original[:4]
p2=["P00",*original][4:8]
assert len(set(p1+p2))==7 and set(original)-set(p1+p2)=={"P08"}
print("Inserción entre páginas:",p1,p2,"ausente P08; repetido P04")


## A6 · Fragmentación, reducción y medición
Leer por bloques mantiene un estado parcial. Para conteos por entidad, acumula un Counter; el tamaño de bloque cambia la cantidad de llamadas, no el resultado si el universo y la operación son iguales. Para media, combina suma y conteo de valores válidos. Una media de medias da peso a los grupos, no a las observaciones. La mediana exacta no se obtiene así.

El benchmark genera un CSV sintético de dos columnas. Para la misma N, mide lectura completa y fragmentada en tres configuraciones. El archivo y el DataFrame ocupan cantidades diferentes. memory_usage(deep=True) estima tamaño lógico de columnas e índice, no pico RSS. perf_counter mide tiempo transcurrido, no tiempo de CPU. No concluyas que una estrategia es universalmente más rápida por una sola ejecución; la caché y el entorno pueden cambiar tiempos. No se ejecutan cargas grandes por defecto.

**Predice:** ¿qué ocurre al cambiar de 1000 a 5000 filas por bloque? ¿Debe cambiar la suma? Registra resultado y máxima memoria lógica de un bloque. Las pruebas de igualdad deben pasar antes de comparar costo.


In [ ]:
def contar_bloques(filas,chunk):
    assert chunk>0
    out=Counter(); tamanos=[]
    for i in range(0,len(filas),chunk):
        lote=filas[i:i+chunk]; tamanos.append(len(lote))
        out.update(r["entidad"] for r in lote)
    return out,tamanos
directo=Counter(r["entidad"] for r in raw)
for b in (7,25,50):
    parcial,tamanos=contar_bloques(raw,b)
    assert parcial==directo and sum(tamanos)==len(raw)
    print("bloque",b,"particiones",len(tamanos),"paridad OK")
a=[10,10,None]; b=[100,100,100,100]
estados=[(sum(x for x in z if x is not None),sum(x is not None for x in z)) for z in (a,b)]
media=sum(x[0] for x in estados)/sum(x[1] for x in estados)
assert media==70
print("Estados",estados,"media global",media,"media de medias",55)


In [ ]:
def medir_csv(n,chunk_sizes):
    resultados=[]
    with tempfile.TemporaryDirectory(prefix="s01_medicion_") as tmp:
        p=Path(tmp)/"sintetico.csv"
        with p.open("w",newline="",encoding="utf-8") as f:
            w=csv.writer(f);w.writerow(["id","valor"])
            w.writerows((i,i%101) for i in range(n))
        t=time.perf_counter(); d=pd.read_csv(p)
        ref=(len(d),int(d.valor.sum())); t_full=time.perf_counter()-t
        resultados.append({"N":n,"chunk":"completo","filas":ref[0],"suma":ref[1],
                           "bytes_logicos":int(d.memory_usage(deep=True).sum()),"bytes_csv":p.stat().st_size,"segundos":t_full})
        del d
        for c in chunk_sizes:
            t=time.perf_counter(); cnt=total=mem=0
            for part in pd.read_csv(p,chunksize=c):
                cnt+=len(part);total+=int(part.valor.sum())
                mem=max(mem,int(part.memory_usage(deep=True).sum()))
            assert (cnt,total)==ref
            resultados.append({"N":n,"chunk":c,"filas":cnt,"suma":total,"bytes_logicos":mem,
                               "bytes_csv":p.stat().st_size,"segundos":time.perf_counter()-t})
    return resultados
mediciones=medir_csv(10_000,[500,1000,5000])
display(pd.DataFrame(mediciones))
assert len({r["suma"] for r in mediciones})==1


## A7 · Producto RAW, derivado y manifiesto
RAW preserva los registros adquiridos sin alterar campos; JSONL guarda un objeto por línea. Un derivado es el conteo por entidad. El manifiesto conserva fuente, versión de blob, hash canónico del snapshot, grano, esquema, parámetros, traza, controles y SHA-256 de archivos. La carpeta nueva se crea con tempfile.mkdtemp y no se elimina al cerrar la celda; dura en el runtime. En Colab puedes descargarla desde el explorador de archivos. No se escribe en el repositorio.

Una huella verifica bytes, no veracidad ni autoría. Por eso se comprueba también contenido por ID, ausencia de duplicados y equivalencia de conteos al releer. Una modificación de un campo conserva identidad pero cambia bytes y contenido. Una inversión de orden puede cambiar bytes sin alterar contenido por ID. No se guarda información de credenciales.

**Producto mínimo:** carpeta con snapshot.json, raw.jsonl, conteos.json y manifest.json; relectura satisfactoria. La fuente completa se conserva separada de la adquisición seleccionada. El manifiesto indica si el universo elegido es un prefijo del snapshot, no lo llama cobertura nacional.


In [ ]:
def producir(base,limite,chunk,universo="snapshot completo"):
    filas,ctrl,tr=adquirir(base,limite,limite,math.ceil(len(base)/limite))
    assert ctrl["unicos"]==len(base) and not ctrl["duplicados"] and not ctrl["ausentes"] and not ctrl["extras"]
    agg,partes=contar_bloques(filas,chunk)
    assert agg==Counter(r["entidad"] for r in base)
    destino=Path(tempfile.mkdtemp(prefix="s01_producto_"))
    (destino/"snapshot.json").write_bytes(canon(CORPUS))
    (destino/"raw.jsonl").write_bytes(b"".join(canon(r)+b"\n" for r in filas))
    (destino/"conteos.json").write_bytes(canon(dict(agg)))
    man={"schema":"s01-producto-v1","fuente":FUENTE,"blob_fuente":BLOB_FUENTE,"hash_snapshot":HASH_SNAPSHOT,
         "grano":"proceso","orden":"posición en snapshot versionado","universo":universo,"total_snapshot":len(CORPUS),"esquema":ESQUEMA,
         "limit":limite,"chunk":chunk,"traza":tr,"bloques":partes,"control":ctrl,
         "archivos":{p.name:sha(p.read_bytes()) for p in destino.iterdir()}}
    (destino/"manifest.json").write_bytes(canon(man))
    leido=json.loads((destino/"manifest.json").read_bytes())
    assert all(sha((destino/k).read_bytes())==v for k,v in leido["archivos"].items())
    releido=[json.loads(l) for l in (destino/"raw.jsonl").read_bytes().splitlines()]
    assert releido==filas and len({r["id_proceso"] for r in releido})==len(base)
    assert json.loads((destino/"conteos.json").read_bytes())==dict(Counter(r["entidad"] for r in releido))
    assert sha((destino/"snapshot.json").read_bytes())==HASH_SNAPSHOT
    return destino,man
destino,man=producir(base,37,25)
print("Paquete comprobado:",destino)
print({k:man[k] for k in ("grano","universo","control","archivos")})


## A8 · Transferencia y defensa
Cambia límite y chunk, conserva universo. Predice qué propiedades permanecen invariantes: los IDs, contenido por ID y conteos. El número de páginas cambia. Luego reduce el universo a los primeros 80 procesos del snapshot en su orden versionado y etiqueta explícitamente la selección. Esta segunda operación cambia la población; no esperes el mismo conteo. No se debe confundir cobertura del prefijo con cobertura del snapshot completo.

El reto no tiene una respuesta de memoria: el resultado depende de los parámetros. Antes de ejecutar, explica qué propiedad comprobarás; después contrasta el manifiesto y la relectura. La evidencia para la defensa incluye dos productos, configuración y explicación del límite. Una extracción estática no acredita frescura del portal ni irregularidades.

**Salida para el futuro S02:** fuente, grano, cantidad observada, esquema, identidad, reglas de adquisición y restricciones. La tasa de actualización de SECOP y el SLA siguen desconocidos en este laboratorio; no se inventan. La arquitectura futura debe derivarse de requisitos, no de una marca.


In [ ]:
# Cambia estos valores para practicar, antes de ejecutar predice qué se conserva.
N_RETO=80
LIMIT_RETO=19
CHUNK_RETO=13
assert 1<=N_RETO<=len(base)
d2,m2=producir(base,LIMIT_RETO,CHUNK_RETO)
assert m2["archivos"]["raw.jsonl"]==man["archivos"]["raw.jsonl"]
assert m2["archivos"]["conteos.json"]==man["archivos"]["conteos.json"]
d3,m3=producir(base[:N_RETO],LIMIT_RETO,CHUNK_RETO,universo=f"prefijo {N_RETO} de {len(base)}")
assert m3["control"]["unicos"]==N_RETO
print("Mismo universo, mismos bytes RAW:",d2)
print("Universo distinto, cobertura declarada:",d3,m3["universo"])
print("RELEVO",{"fuente":FUENTE,"grano":"proceso","filas":len(base),"esquema":ESQUEMA,
                "frecuencia":"no medida","SLA":"no definido"})
